# GPU G1 — Hybrid runtime parity and performance audit

Bounded Kaggle T4 validation only; it does not change BTC mapping, CLIP model space, OPUS revision, Stage1 exact NumPy search, or RT2 semantics.

Required Kaggle inputs:
1. Raw AIC dataset: `/kaggle/input/datasets/nadkli/dataset-aic`
2. Stage1 index: `/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle`
3. Stage1B verification: `/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports`
4. Stage1E language freeze: `/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze`
5. Offline OpenAI CLIP: `/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32`
6. Offline OPUS vi-en: `/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en`
7. Optional NVDEC wheel: `/kaggle/input/datasets/irthn1311/aic2026-pynvvideocodec-wheel`. Nested roots are supported; override with `AIC_PYNVVIDEOCODEC_WHEEL_ROOT`. Without it, NVDEC is reported `UNAVAILABLE` and the audit still completes on OpenCV.

Internet is used only to clone/update repository code when it is absent; all model inference is local-only. Attach a repository snapshot and set `AIC_REPO_DIR` for a fully offline run. Output: `/kaggle/working/triage_eg_gpu_g1_bundle.zip`.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time
import numpy as np

REPO_URL = os.environ.get('AIC_REPO_URL', 'https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git')
REPO_REF = os.environ.get('AIC_REPO_REF', 'TRIAGEEG')
REPO_DIR = Path(os.environ.get('AIC_REPO_DIR', '/kaggle/working/AIC2026_TeamPTK_SGU'))
if not (REPO_DIR / 'src/triage_eg').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
wheel_root = Path(os.environ.get('AIC_PYNVVIDEOCODEC_WHEEL_ROOT', '/kaggle/input/datasets/irthn1311/aic2026-pynvvideocodec-wheel'))
wheel_name = 'pynvvideocodec-2.1.0-cp312-cp312-manylinux_2_28_x86_64.whl'
wheel_candidates = sorted(wheel_root.rglob(wheel_name)) if wheel_root.exists() else []
if wheel_candidates:
    if sys.version_info[:2] != (3, 12): raise RuntimeError(f'PyNvVideoCodec wheel requires CPython 3.12, found {sys.version}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', str(wheel_candidates[0])], check=True)
    print('offline PyNvVideoCodec wheel:', wheel_candidates[0])
else:
    print('optional PyNvVideoCodec wheel not attached; NVDEC will be UNAVAILABLE')
sys.path.insert(0, str(REPO_DIR / 'src'))
COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
OUTPUT_ROOT = Path('/kaggle/working/triage_eg_gpu_g1')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print({'ref': REPO_REF, 'commit': COMMIT, 'output': str(OUTPUT_ROOT)})

In [ ]:
from triage_eg.video.g1_audit import representative_videos

def resolve_root(requested, marker, max_depth=7):
    requested = Path(requested)
    if (requested / marker).exists():
        return requested.resolve()
    candidates = []
    for base in (requested, Path('/kaggle/input')):
        if not base.exists():
            continue
        for found in base.rglob(Path(marker).name):
            if len(found.relative_to(base).parts) > max_depth or not found.is_file():
                continue
            root = found.parents[len(Path(marker).parts) - 1]
            if (root / marker).is_file():
                candidates.append(root.resolve())
    unique = sorted(set(candidates), key=lambda value: (len(value.parts), value.as_posix()))
    if not unique:
        raise FileNotFoundError(f'Cannot resolve {marker} below {requested}')
    return unique[0]

DATA_ROOT = Path(os.environ.get('AIC_DATA_ROOT', '/kaggle/input/datasets/nadkli/dataset-aic'))
STAGE1_ROOT = resolve_root(os.environ.get('AIC_STAGE1_ROOT', '/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle'), 'index/clip_vectors.f16.npy')
STAGE1B_ROOT = resolve_root(os.environ.get('AIC_STAGE1B_ROOT', '/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports'), 'encoder/selected_encoder_contract.json')
STAGE1E_ROOT = resolve_root(os.environ.get('AIC_STAGE1E_ROOT', '/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze'), 'language_path_contract.json')
CLIP_ROOT = resolve_root(os.environ.get('AIC_CLIP_ROOT', '/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32'), 'checkpoint/ViT-B-32.pt')
OPUS_ROOT = resolve_root(os.environ.get('AIC_OPUS_ROOT', '/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en'), 'model/config.json')
VIDEOS = representative_videos(DATA_ROOT, limit=int(os.environ.get('AIC_G1_VIDEO_LIMIT', '4')))
INPUTS = {'dataset': str(DATA_ROOT), 'stage1': str(STAGE1_ROOT), 'stage1b': str(STAGE1B_ROOT), 'stage1e': str(STAGE1E_ROOT), 'clip': str(CLIP_ROOT), 'opus': str(OPUS_ROOT), 'videos': [str(path) for path in VIDEOS]}
print(json.dumps(INPUTS, indent=2))

In [ ]:
import platform
import torch
from triage_eg.video import HardwareConfig, nvdec_preflight, resolve_hardware

NVDEC_PREFLIGHT = nvdec_preflight()
HARDWARE = resolve_hardware(HardwareConfig(), torch_module=torch, nvdec_probe=NVDEC_PREFLIGHT).as_dict()
GPU_PREFLIGHT = {'python': platform.python_version(), 'torch': torch.__version__, 'torch_cuda': getattr(torch.version, 'cuda', None), 'cuda_available': torch.cuda.is_available(), 'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, 'nvdec': NVDEC_PREFLIGHT, 'effective_auto': HARDWARE}
print(json.dumps(GPU_PREFLIGHT, indent=2))

In [ ]:
from triage_eg.video.g1_audit import benchmark_decoder_paths

DECODER_BENCHMARK, DECODER_PARITY = benchmark_decoder_paths(VIDEOS, nvdec_available=bool(NVDEC_PREFLIGHT['available']))
print('decoder records:', len(DECODER_BENCHMARK['records']))
print('raw frame identity:', DECODER_PARITY['identity_status'])

In [ ]:
from dataclasses import replace
from triage_eg.retrieval.stage2 import OperationalRetrievalRuntime, QueryRequest, config_from_yaml
from triage_eg.video import OpenCVRawVideoDecoder
from triage_eg.video.g1_audit import audit_indices, retrieval_parity, vector_parity

queries = [QueryRequest('g1_en_car', 'a red car', 'en', 50), QueryRequest('g1_en_cook', 'a person cooking in a kitchen', 'en', 50), QueryRequest('g1_vi_car', 'một chiếc ô tô màu đỏ', 'vi', 50), QueryRequest('g1_vi_cook', 'một người đang nấu ăn trong bếp', 'vi', 50), QueryRequest('g1_vi_soccer', 'mọi người đang chơi bóng đá', 'vi', 50)]
base_kwargs = dict(stage1_root=STAGE1_ROOT, stage1b_root=STAGE1B_ROOT, stage1e_root=STAGE1E_ROOT, clip_asset_root=CLIP_ROOT, translator_asset_root=OPUS_ROOT, stage1d_config=REPO_DIR / 'configs/retrieval/stage1d_translation_ablation.yaml', build_git_commit=COMMIT)
frame_decoder = OpenCVRawVideoDecoder(VIDEOS[0].stem, VIDEOS[0])
frame_ids = audit_indices(frame_decoder.info.total_frames)[:8]
rgb_frames = [frame.image for frame in frame_decoder.decode_indices(frame_ids)]
frame_decoder.close()
SHARED_RUNTIME_ROOT = OUTPUT_ROOT / '_shared_stage2_runtime'
cpu_config = config_from_yaml(REPO_DIR / 'configs/retrieval/stage2_operational_runtime.yaml', output_root=SHARED_RUNTIME_ROOT, **base_kwargs)
cpu_runtime = OperationalRetrievalRuntime(cpu_config).load()
started = time.monotonic(); cpu_images = cpu_runtime.encoder.encode_rgb_arrays(rgb_frames); cpu_image_ms = (time.monotonic() - started) * 1000
started = time.monotonic(); cpu_batch = cpu_runtime.encode_requests(queries); cpu_query_ms = (time.monotonic() - started) * 1000
clip_inputs = [item['clip_input_text'] for item in cpu_batch.encodings]
started = time.monotonic(); cpu_clip_text = cpu_runtime.encoder.encode_text(clip_inputs); cpu_clip_text_ms = (time.monotonic() - started) * 1000
vi_texts = [query.text for query in queries if query.language == 'vi']
started = time.monotonic(); cpu_translation_rows = cpu_runtime.translator.translate(vi_texts); cpu_translation_ms = (time.monotonic() - started) * 1000
cpu_translations = [item['translated_text_for_clip'] for item in cpu_translation_rows]
cpu_manifest = cpu_runtime.runtime_manifest()
cpu_runtime.close()
if torch.cuda.is_available():
    gpu_config = config_from_yaml(REPO_DIR / 'configs/retrieval/stage2_operational_runtime_gpu.yaml', output_root=SHARED_RUNTIME_ROOT, **base_kwargs)
    gpu_runtime = OperationalRetrievalRuntime(gpu_config).load()
    started = time.monotonic(); gpu_images = gpu_runtime.encoder.encode_rgb_arrays(rgb_frames); gpu_image_ms = (time.monotonic() - started) * 1000
    started = time.monotonic(); gpu_batch = gpu_runtime.encode_requests(queries); gpu_query_ms = (time.monotonic() - started) * 1000
    started = time.monotonic(); gpu_clip_text = gpu_runtime.encoder.encode_text(clip_inputs); gpu_clip_text_ms = (time.monotonic() - started) * 1000
    started = time.monotonic(); gpu_translation_rows = gpu_runtime.translator.translate(vi_texts); gpu_translation_ms = (time.monotonic() - started) * 1000
    gpu_translations = [item['translated_text_for_clip'] for item in gpu_translation_rows]
    gpu_manifest = gpu_runtime.runtime_manifest()
    index_vectors = np.load(STAGE1_ROOT / 'index/clip_vectors.f16.npy', mmap_mode='r')
    CLIP_PARITY = vector_parity(cpu_clip_text, gpu_clip_text, top_k=len(queries))
    CLIP_PARITY['retrieval'] = retrieval_parity(cpu_clip_text, gpu_clip_text, index_vectors, top_k=50)
    CLIP_PARITY['image_embeddings'] = vector_parity(cpu_images, gpu_images, top_k=len(rgb_frames))
    if CLIP_PARITY['retrieval']['status'] != 'PASS' or CLIP_PARITY['image_embeddings']['status'] != 'PASS': CLIP_PARITY['status'] = 'FAIL'
    CLIP_PARITY['performance'] = {'cpu_image_ms': cpu_image_ms, 'gpu_image_ms': gpu_image_ms, 'image_speedup': cpu_image_ms / gpu_image_ms if gpu_image_ms else None, 'cpu_text_ms': cpu_clip_text_ms, 'gpu_text_ms': gpu_clip_text_ms, 'text_speedup': cpu_clip_text_ms / gpu_clip_text_ms if gpu_clip_text_ms else None, 'cpu_end_to_end_query_ms': cpu_query_ms, 'gpu_end_to_end_query_ms': gpu_query_ms}
    TRANSLATOR_PARITY = {'status': 'PASS' if cpu_translations == gpu_translations else 'FAIL', 'cpu': cpu_translations, 'gpu': gpu_translations, 'performance': {'cpu_translation_ms': cpu_translation_ms, 'gpu_translation_ms': gpu_translation_ms, 'speedup': cpu_translation_ms / gpu_translation_ms if gpu_translation_ms else None}}
    gpu_runtime.close()
else:
    CLIP_PARITY = {'status': 'UNAVAILABLE', 'reason': 'CUDA_UNAVAILABLE', 'cpu_image_ms': cpu_image_ms, 'cpu_query_ms': cpu_query_ms}
    TRANSLATOR_PARITY = {'status': 'UNAVAILABLE', 'reason': 'CUDA_UNAVAILABLE', 'cpu': cpu_translations, 'performance': {'cpu_translation_ms': cpu_translation_ms}}
    gpu_manifest = None
print(json.dumps({'clip': CLIP_PARITY, 'translator': TRANSLATOR_PARITY}, indent=2))

In [ ]:
from datetime import UTC, datetime
from triage_eg.video.g1_audit import build_g1_decision

DECISION = build_g1_decision(DECODER_BENCHMARK, DECODER_PARITY, CLIP_PARITY, TRANSLATOR_PARITY, cuda_available=torch.cuda.is_available(), nvdec_available=bool(NVDEC_PREFLIGHT['available']))
PERFORMANCE = {'decoder': DECODER_BENCHMARK, 'clip': CLIP_PARITY.get('performance', {}), 'translator': TRANSLATOR_PARITY.get('performance', {}), 'runtime_manifests': {'cpu': cpu_manifest, 'gpu': gpu_manifest}, 'decision_speedups': DECISION['speedups']}
RUN_MANIFEST = {'sprint': 'GPU_G1', 'created_at': datetime.now(UTC).isoformat(), 'git_commit': COMMIT, 'inputs': INPUTS, 'hardware': GPU_PREFLIGHT, 'frozen_contracts': {'btc_mapping': 'UNCHANGED', 'clip_model_space': 'OFFICIAL_OPENAI_VIT_B32_UNCHANGED', 'vi_path': 'OPUS_MT_THEN_CLIP_UNCHANGED', 'stage1': 'NUMPY_EXACT_UNCHANGED', 'rt2': 'ORDER_ONLY_MONOTONIC_DP_LAMBDA_0_UNCHANGED'}, 'decision': DECISION}
ISSUES = []
if not NVDEC_PREFLIGHT['available']: ISSUES.append({'severity': 'INFO', 'code': 'NVDEC_UNAVAILABLE', 'detail': NVDEC_PREFLIGHT.get('reason')})
if DECODER_PARITY['identity_status'] == 'FAIL': ISSUES.append({'severity': 'ERROR', 'code': 'RAW_FRAME_IDENTITY_FAILED'})
print(json.dumps(DECISION, indent=2))

In [ ]:
from triage_eg.video.g1_audit import write_g1_bundle

ARTIFACTS = {'gpu_preflight.json': GPU_PREFLIGHT, 'decoder_benchmark.json': DECODER_BENCHMARK, 'decoder_parity.json': DECODER_PARITY, 'clip_cpu_gpu_parity.json': CLIP_PARITY, 'translator_cpu_gpu_parity.json': TRANSLATOR_PARITY, 'performance_summary.json': PERFORMANCE, 'run_manifest.json': RUN_MANIFEST, 'issues.jsonl': ISSUES}
ZIP_PATH = write_g1_bundle(OUTPUT_ROOT, ARTIFACTS)
for key in ('GPU_G1_STATUS', 'CPU_FALLBACK_STATUS', 'OPTIMIZED_OPENCV_STATUS', 'NVDEC_STATUS', 'CLIP_GPU_STATUS', 'TRANSLATOR_GPU_STATUS', 'M1_IN_MEMORY_CLIP_STATUS', 'DEFAULT_VIDEO_BACKEND', 'MAIN_RUNTIME_GPU_AWARE'):
    print(f'{key}={DECISION[key]}')
print('DOWNLOAD_ZIP=' + str(ZIP_PATH))